In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%cd /content

!rm -rf /content/smallnet
!git clone https://github.com/SepehrAkbari/smallnet.git

%cd /content/smallnet

!git status --short
!git log -1 --oneline

/content
Cloning into 'smallnet'...
remote: Enumerating objects: 2150, done.
remote: Counting objects: 100% (265/265), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 2150 (delta 149), reused 171 (delta 64), pack-reused 1885 (from 1)
Receiving objects: 100% (2150/2150), 582.51 MiB | 27.31 MiB/s, done.
Resolving deltas: 100% (384/384), done.
Updating files: 100% (1664/1664), done.
/content/smallnet
869ade6 (HEAD -> main, origin/main, origin/HEAD) restructure + val


In [3]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!ln -sf /root/.local/bin/uv /usr/local/bin/uv

!uv --version
!uv sync

downloading uv 0.11.31 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/bin/bash: line 1: uv: command not found
/bin/bash: line 1: uv: command not found


In [6]:
!uv --version
!uv sync

/bin/bash: line 1: uv: command not found
/bin/bash: line 1: uv: command not found


In [5]:
from pathlib import Path
import subprocess

repo = Path("/content/smallnet")
backup = Path(
    "/content/drive/MyDrive/smallnet_colab_backup"
)

assert backup.exists(), f"Backup directory not found: {backup}"

restore_pairs = [
    (
        backup / "camvid_vgg_cp",
        repo / "results/camvid_vgg_cp",
    ),
    (
        backup / "paper",
        repo / "results/paper",
    ),
]

for source, destination in restore_pairs:
    if source.exists():
        destination.mkdir(parents=True, exist_ok=True)

        subprocess.run(
            [
                "rsync",
                "-a",
                f"{source}/",
                f"{destination}/",
            ],
            check=True,
        )

        print(f"Restored: {source}")
    else:
        print(f"Not found, skipped: {source}")

# Prevent the old accidental duplicate nesting from returning.
duplicate = (
    repo
    / "results/camvid_vgg_cp/camvid_vgg_cp"
)

if duplicate.exists():
    subprocess.run(
        ["rm", "-rf", str(duplicate)],
        check=True,
    )
    print("Removed accidental duplicate nesting.")

Restored: /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp
Restored: /content/drive/MyDrive/smallnet_colab_backup/paper


In [7]:
from pathlib import Path
import shutil

duplicate_backup = Path(
    "/content/drive/MyDrive/"
    "smallnet_colab_backup/"
    "camvid_vgg_cp/camvid_vgg_cp"
)

if duplicate_backup.exists():
    shutil.rmtree(duplicate_backup)
    print("Removed duplicate nesting from Drive backup.")
else:
    print("No duplicate Drive directory found.")

No duplicate Drive directory found.


In [9]:
from pathlib import Path
import shutil
import subprocess

drive_root = Path("/content/drive/MyDrive")
repo = Path("/content/smallnet")

model_candidates = [
    drive_root
    / "smallnet_colab_backup/model/best_model.pth",
    drive_root
    / "smallnet/model/best_model.pth",
]

model_source = next(
    (
        path
        for path in model_candidates
        if path.is_file()
    ),
    None,
)

if model_source is None:
    model_matches = list(
        drive_root.rglob("best_model.pth")
    )
    model_source = (
        model_matches[0]
        if model_matches
        else None
    )

assert model_source is not None, (
    "Could not find best_model.pth in Google Drive."
)

model_destination = (
    repo / "model/best_model.pth"
)
model_destination.parent.mkdir(
    parents=True,
    exist_ok=True,
)
shutil.copy2(
    model_source,
    model_destination,
)

print("Model source:", model_source)
print("Model copied to:", model_destination)

Model source: /content/drive/MyDrive/model/best_model.pth
Model copied to: /content/smallnet/model/best_model.pth


In [10]:
from pathlib import Path

repo = Path("/content/smallnet")

required = [
    repo / "model/best_model.pth",
    repo / "data/CamVid/class_dict.csv",
    repo / "data/CamVid/train",
    repo / "data/CamVid/train_labels",
    repo / "data/CamVid/val",
    repo / "data/CamVid/val_labels",
    repo / "data/CamVid/test",
    repo / "data/CamVid/test_labels",
    repo
    / "results/camvid_vgg_cp/"
    "dataset_validation_report.json",
]

for path in required:
    print(
        "OK" if path.exists() else "MISSING",
        path,
    )

assert all(path.exists() for path in required)

OK /content/smallnet/model/best_model.pth
OK /content/smallnet/data/CamVid/class_dict.csv
OK /content/smallnet/data/CamVid/train
OK /content/smallnet/data/CamVid/train_labels
OK /content/smallnet/data/CamVid/val
OK /content/smallnet/data/CamVid/val_labels
OK /content/smallnet/data/CamVid/test
OK /content/smallnet/data/CamVid/test_labels
OK /content/smallnet/results/camvid_vgg_cp/dataset_validation_report.json


In [11]:
from pathlib import Path

camvid = Path("/content/smallnet/data/CamVid")

for split in ["train", "val", "test"]:
    image_count = len(
        list((camvid / split).glob("*"))
    )
    mask_count = len(
        list(
            (
                camvid / f"{split}_labels"
            ).glob("*")
        )
    )

    print(
        split,
        "images:",
        image_count,
        "masks:",
        mask_count,
    )

train images: 369 masks: 369
val images: 100 masks: 100
test images: 232 masks: 232


In [12]:
!nvidia-smi

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

assert torch.cuda.is_available()

Thu Jul 23 20:16:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
import json
from pathlib import Path

config_path = Path(
    "/content/smallnet/"
    "configs/camvid_vgg_cp_paper.json"
)

config = json.loads(
    config_path.read_text()
)

print(
    json.dumps(
        config.get("final_structural", {}),
        indent=2,
    )
)

{
  "ranks": [
    32,
    64,
    128,
    256,
    512
  ],
  "seeds": [
    0,
    1,
    2
  ],
  "iteration_budget": 200,
  "init": "random",
  "memory_efficient_mttkrp": true,
  "mttkrp_rank_chunk_size": 64,
  "mttkrp_max_explicit_bytes": 536870912,
  "numerical_tolerance": 1e-05,
  "residual_output_chunk_size": 8,
  "factor_diagnostic_thresholds": {
    "near_zero_component_norm_threshold": 1e-12,
    "extreme_factor_norm_threshold": 1000000.0,
    "scaling_spread_threshold": 1000000.0,
    "cancellation_ratio_threshold": 1000.0,
    "bounded_reconstruction_multiple": 10.0
  },
  "output_dir": "results/camvid_vgg_cp/final_structural",
  "figures_dir": "results/paper/figures",
  "audit_path": "results/paper/final_structural_audit.md"
}


In [14]:
%cd /content/smallnet

!uv run python -m pytest -q

/content/smallnet
/bin/bash: line 1: uv: command not found
